# Fusion Model Training

This notebook trains a fusion model that combines predictions from three modalities:
- **Text**: DistilBERT fine-tuned on GoEmotions
- **Audio**: CNN trained on speech features
- **Face**: ViT trained on facial expressions

The fusion model takes probability vectors (6-class) from each modality and learns to combine them into a final 6-class prediction.

**6 Emotion Classes:** Positive, Neutral, Stress, Anxiety, Negative, Depression

In [ ]:
!pip install tensorflow pandas numpy scikit-learn matplotlib seaborn datasets -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.regularizers import l2

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

In [ ]:
CLASS_NAMES = ["Positive", "Neutral", "Stress", "Anxiety", "Negative", "Depression"]
NUM_CLASSES = len(CLASS_NAMES)
NUM_MODALITIES = 3
INPUT_DIM = NUM_CLASSES * NUM_MODALITIES  # 18 features

print(f"Number of classes: {NUM_CLASSES}")
print(f"Input feature dimension: {INPUT_DIM}")

## Data Generation

Since no complete multimodal dataset exists with all three modalities aligned, we simulate training data using:
- **GoEmotions** text dataset for ground truth labels
- Synthetic audio and face probability vectors with varying noise levels
- Controlled scenarios: all agree, two agree, only one informative, etc.

In [ ]:
print("Loading GoEmotions dataset...")
from datasets import load_dataset
ds = load_dataset("google-research-datasets/go_emotions")
train_ds = ds["train"]
test_ds = ds["test"]
print(f"Train set size: {len(train_ds)}")
print(f"Test set size: {len(test_ds)}")

In [ ]:
id2label_28 = {
    0: 'admiration', 1: 'amusement', 2: 'anger', 3: 'annoyance',
    4: 'approval', 5: 'caring', 6: 'confusion', 7: 'curiosity',
    8: 'desire', 9: 'disappointment', 10: 'disapproval', 11: 'disgust',
    12: 'embarrassment', 13: 'excitement', 14: 'fear', 15: 'gratitude',
    16: 'grief', 17: 'joy', 18: 'love', 19: 'nervousness',
    20: 'optimism', 21: 'pride', 22: 'realization', 23: 'relief',
    24: 'remorse', 25: 'sadness', 26: 'surprise', 27: 'neutral'
}

final_map = {
    'admiration': 0, 'amusement': 0, 'approval': 0, 'caring': 0,
    'desire': 0, 'excitement': 0, 'gratitude': 0, 'joy': 0, 'love': 0,
    'optimism': 0, 'pride': 0, 'relief': 0,
    'curiosity': 1, 'realization': 1, 'surprise': 1, 'neutral': 1,
    'anger': 2, 'annoyance': 2, 'disapproval': 2, 'confusion': 2,
    'fear': 3, 'nervousness': 3,
    'disappointment': 4, 'disgust': 4, 'embarrassment': 4, 'remorse': 4,
    'grief': 5, 'sadness': 5
}

def convert_label(example):
    labels = example['labels']
    if len(labels) == 0:
        example['label_6'] = 1
    else:
        first = labels[0]
        example['label_6'] = final_map[id2label_28[first]]
    return example

train_ds = train_ds.map(convert_label)
test_ds = test_ds.map(convert_label)

In [ ]:
def to_one_hot(label, num_classes):
    vec = np.zeros(num_classes)
    vec[label] = 1.0
    return vec

y_train = np.array([to_one_hot(l, NUM_CLASSES) for l in train_ds['label_6']])
y_test = np.array([to_one_hot(l, NUM_CLASSES) for l in test_ds['label_6']])

print(f'y_train shape: {y_train.shape}')
print(f'y_test shape: {y_test.shape}')

In [ ]:
def generate_synthetic_modality_probs(true_label, noise_level=0.3, uncertainty=0.1):
    probs = np.ones(NUM_CLASSES) * uncertainty / (NUM_CLASSES - 1)
    probs[true_label] = 1.0 - uncertainty
    probs += np.random.normal(0, noise_level, NUM_CLASSES)
    probs = np.clip(probs, 0, 1)
    return probs / probs.sum()

def generate_training_sample(true_label, scenario='agree'):
    if scenario == 'agree':
        tp = generate_synthetic_modality_probs(true_label, 0.15)
        ap = generate_synthetic_modality_probs(true_label, 0.20)
        fp = generate_synthetic_modality_probs(true_label, 0.25)
    elif scenario == 'text_dominant':
        tp = generate_synthetic_modality_probs(true_label, 0.1)
        ap = generate_synthetic_modality_probs(true_label, 0.45)
        fp = generate_synthetic_modality_probs(true_label, 0.45)
    elif scenario == 'conflict':
        wrong = (true_label + np.random.randint(1, NUM_CLASSES)) % NUM_CLASSES
        tp = generate_synthetic_modality_probs(true_label, 0.1)
        ap = generate_synthetic_modality_probs(wrong, 0.15)
        fp = generate_synthetic_modality_probs(wrong, 0.15)
    elif scenario == 'uncertain':
        tp = generate_synthetic_modality_probs(true_label, 0.4, 0.5)
        ap = generate_synthetic_modality_probs(true_label, 0.4, 0.5)
        fp = generate_synthetic_modality_probs(true_label, 0.4, 0.5)
    else:
        raise ValueError(f'Unknown scenario: {scenario}')
    return np.concatenate([tp, ap, fp])

In [ ]:
SEED = 42
np.random.seed(SEED)

scenarios = ['agree', 'agree', 'text_dominant', 'conflict', 'uncertain']
scenario_weights = [0.4, 0.3, 0.15, 0.1, 0.05]

def generate_dataset(labels, size):
    X_list = []
    for i in range(size):
        s = np.random.choice(scenarios, p=scenario_weights)
        X_list.append(generate_training_sample(labels[i], s))
    return np.array(X_list)

X_train = generate_dataset(train_ds['label_6'], len(train_ds))
X_test = generate_dataset(test_ds['label_6'], len(test_ds))

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_test shape: {y_test.shape}')

## Model Architecture

A multi-layer perceptron that learns to combine modality probability vectors:
- Input: 18 features (6 probs x 3 modalities: text, audio, face)
- Hidden layers with BatchNorm, Dropout, and L2 regularization
- Output: 6-class softmax

In [ ]:
def build_fusion_model(input_dim=INPUT_DIM, num_classes=NUM_CLASSES):
    inputs = Input(shape=(input_dim,), name='modality_probs')
    x = Dense(128, activation='relu', kernel_regularizer=l2(1e-4))(inputs)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    x = Dense(64, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    x = Dense(32, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    outputs = Dense(num_classes, activation='softmax', name='emotion_probs')(x)
    model = Model(inputs=inputs, outputs=outputs, name='fusion_model')
    return model

model = build_fusion_model()
model.summary()

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss=CategoricalCrossentropy(from_logits=True, label_smoothing=0.1),
    metrics=['accuracy']
)

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    ModelCheckpoint(filepath='../models/fusion_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1)
]

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history.history['loss'], label='Train Loss')
ax1.plot(history.history['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True)
ax2.plot(history.history['accuracy'], label='Train Acc')
ax2.plot(history.history['val_accuracy'], label='Val Acc')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True)
plt.tight_layout()
plt.savefig('../models/fusion_training_history.png', dpi=150)
plt.show()

In [ ]:
y_pred_probs = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

metrics = {}
metrics['Accuracy'] = round(accuracy_score(y_true, y_pred), 4)
metrics['Precision'] = round(precision_score(y_true, y_pred, average='weighted'), 4)
metrics['Recall'] = round(recall_score(y_true, y_pred, average='weighted'), 4)
metrics['F1-Score'] = round(f1_score(y_true, y_pred, average='weighted'), 4)

pd.DataFrame([metrics])

In [ ]:
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Fusion Model Confusion Matrix')
plt.tight_layout()
plt.savefig('../models/fusion_confusion_matrix.png', dpi=150)
plt.show()

## Evaluation Scenarios

Evaluate the fusion model under different modality availability conditions:

In [ ]:
def evaluate_scenario(X_text, X_audio, X_face, y_true, name):
    X = np.concatenate([X_text, X_audio, X_face], axis=1)
    yp = np.argmax(model.predict(X, verbose=0), axis=1)
    acc = accuracy_score(y_true, yp)
    f1 = f1_score(y_true, yp, average='weighted')
    print(f'{name}: Acc={acc:.4f}, F1={f1:.4f}')
    return acc, f1

In [ ]:
print('='*60)
print('EVALUATION SCENARIOS')
print('='*60+chr(10))

y_true = np.argmax(y_test, axis=1)
Xt = X_test[:, :6]
Xa = X_test[:, 6:12]
Xf = X_test[:, 12:18]

evaluate_scenario(Xt, Xa, Xf, y_true, 'A: All 3 modalities')

u = np.ones((len(X_test), 6)) / 6.0
evaluate_scenario(Xt, u, u, y_true, 'B: Text only')
evaluate_scenario(Xt, Xa, u, y_true, 'C: Text+audio')
evaluate_scenario(Xt, u, Xf, y_true, 'D: Text+face')

In [ ]:
model.save('../models/fusion_model.h5')
print('Fusion model saved to: ../models/fusion_model.h5')

In [ ]:
from tensorflow.keras.models import load_model
loaded = load_model('../models/fusion_model.h5', custom_objects={'CategoricalCrossentropy': CategoricalCrossentropy})
test_sample = np.array([generate_training_sample(0, 'agree')])
pred = loaded.predict(test_sample, verbose=0)[0]
pc = CLASS_NAMES[np.argmax(pred)]
print(f'Sanity check: predicted {pc} with confidence {np.max(pred):.4f}')

In [ ]:
print('Done! Fusion model training complete.')